# Pytest — Testing

In [ ]:
# Day 35 — Pytest
# We've built so much: ML models (Day 27-29), saved them (Day 30),
# tracked experiments (Day 32), versioned data (Day 33),
# and created APIs (Day 34). But how do we know it all WORKS correctly?
#
# Answer: TESTS!
#
# Analogy:
# Code without tests = driving a car without brakes
# You might be going fast, but you can't stop safely when something goes wrong
#
# In DevOps, tests are CRITICAL:
#   CI/CD pipeline: code push → run tests → if pass → deploy
#   No tests = no confidence = no automatic deployment
#
# Pytest is Python's MOST POPULAR testing framework because:
#   - Simple syntax (just use assert)
#   - Auto-discovers tests (any file starting with test_)
#   - Rich plugins (coverage, fixtures, parametrize)
#   - Better error messages than unittest
#
# Today we'll cover:
#   1. First test (assert, test_ naming)
#   2. Testing functions and classes
#   3. Testing exceptions (pytest.raises)
#   4. Fixtures (setup/teardown, reusable data)
#   5. Parametrize (test many inputs at once)
#   6. Testing ML code (models, pipelines, data)
#   7. Mocking (unittest.mock)
#
# Install: pip install pytest (already in most setups)
#
# NOTE: Pytest runs from the COMMAND LINE, not inside Jupyter.
# We'll write test files and run them with !pytest

### Your First Test

In [ ]:
# Pytest rules:
# 1. Test files must start with test_ (e.g., test_calculator.py)
# 2. Test functions must start with test_ (e.g., def test_add())
# 3. Use assert to check if things are correct
# 4. Run with: pytest (auto-discovers and runs all tests)

import os
os.makedirs('pytest_demo', exist_ok=True)

# First, create a module to test
module_code = '''def add(a, b):
    """Add two numbers."""
    return a + b

def subtract(a, b):
    """Subtract b from a."""
    return a - b

def multiply(a, b):
    """Multiply two numbers."""
    return a * b

def divide(a, b):
    """Divide a by b. Raises ValueError if b is 0."""
    if b == 0:
        raise ValueError("Cannot divide by zero!")
    return a / b
'''

with open('pytest_demo/calculator.py', 'w') as f:
    f.write(module_code)

# Now write the tests
test_code = '''from calculator import add, subtract, multiply, divide

# Test 1: Basic addition
def test_add():
    assert add(2, 3) == 5

# Test 2: Addition with negatives
def test_add_negative():
    assert add(-1, -1) == -2
    assert add(-1, 1) == 0

# Test 3: Subtraction
def test_subtract():
    assert subtract(10, 3) == 7

# Test 4: Multiply
def test_multiply():
    assert multiply(4, 5) == 20
    assert multiply(0, 100) == 0

# Test 5: Division
def test_divide():
    assert divide(10, 2) == 5.0
    assert divide(7, 2) == 3.5
'''

with open('pytest_demo/test_calculator.py', 'w') as f:
    f.write(test_code)

print('Created:')
print('  pytest_demo/calculator.py       ← the code to test')
print('  pytest_demo/test_calculator.py   ← the tests')
print()
print('To run: cd pytest_demo && pytest -v')
print('  -v = verbose (shows each test name and result)')

In [ ]:
# Let's run the tests!
# The ! prefix runs shell commands from Jupyter

import subprocess
result = subprocess.run(
    ['python', '-m', 'pytest', 'pytest_demo/test_calculator.py', '-v'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

### Testing Exceptions with pytest.raises

In [ ]:
# Sometimes you WANT code to raise an error
# divide(10, 0) SHOULD raise ValueError
# pytest.raises() verifies that the exception happens

test_exceptions = '''import pytest
from calculator import divide

# Test that division by zero raises ValueError
def test_divide_by_zero():
    with pytest.raises(ValueError):
        divide(10, 0)

# You can also check the error message
def test_divide_by_zero_message():
    with pytest.raises(ValueError, match="Cannot divide by zero"):
        divide(10, 0)

# Test that valid division does NOT raise
def test_divide_valid():
    result = divide(10, 2)
    assert result == 5.0
    assert isinstance(result, float)
'''

with open('pytest_demo/test_exceptions.py', 'w') as f:
    f.write(test_exceptions)

# Run them
result = subprocess.run(
    ['python', '-m', 'pytest', 'pytest_demo/test_exceptions.py', '-v'],
    capture_output=True, text=True
)
print(result.stdout)

# KEY POINT: pytest.raises is like a try/except for tests
# It PASSES if the exception is raised, FAILS if it's not

### Fixtures — Reusable Test Data

In [ ]:
# Fixtures = reusable setup for your tests
#
# Problem: many tests need the SAME data or objects
# Without fixtures: copy-paste setup code in every test
# With fixtures: write once, inject into any test
#
# Analogy:
# Without fixtures = every test sets up its own kitchen from scratch
# With fixtures = there's a shared kitchen, every test just walks in and cooks

# First, create a server monitoring module
server_module = '''class ServerMonitor:
    def __init__(self):
        self.servers = {}

    def add_server(self, name, cpu, memory, status="healthy"):
        self.servers[name] = {"cpu": cpu, "memory": memory, "status": status}

    def get_server(self, name):
        if name not in self.servers:
            raise KeyError(f"Server {name} not found")
        return self.servers[name]

    def get_critical_servers(self):
        return {name: info for name, info in self.servers.items()
                if info["cpu"] > 90 or info["memory"] > 90}

    def server_count(self):
        return len(self.servers)

    def average_cpu(self):
        if not self.servers:
            return 0
        return sum(s["cpu"] for s in self.servers.values()) / len(self.servers)
'''

with open('pytest_demo/server_monitor.py', 'w') as f:
    f.write(server_module)

# Tests WITH fixtures
test_fixtures = '''import pytest
from server_monitor import ServerMonitor

# --- FIXTURES ---

@pytest.fixture
def monitor():
    """Create a fresh ServerMonitor for each test."""
    return ServerMonitor()

@pytest.fixture
def loaded_monitor():
    """Create a ServerMonitor with sample data."""
    mon = ServerMonitor()
    mon.add_server("web-01", cpu=45, memory=60)
    mon.add_server("web-02", cpu=92, memory=88)
    mon.add_server("db-01",  cpu=30, memory=95)
    return mon

# --- TESTS ---

def test_add_server(monitor):    # ← fixture injected automatically!
    monitor.add_server("web-01", cpu=50, memory=60)
    assert monitor.server_count() == 1
    assert monitor.get_server("web-01")["cpu"] == 50

def test_server_count(loaded_monitor):    # ← uses pre-loaded fixture
    assert loaded_monitor.server_count() == 3

def test_critical_servers(loaded_monitor):
    critical = loaded_monitor.get_critical_servers()
    assert "web-02" in critical    # cpu=92 > 90
    assert "db-01" in critical     # memory=95 > 90
    assert "web-01" not in critical

def test_average_cpu(loaded_monitor):
    avg = loaded_monitor.average_cpu()
    expected = (45 + 92 + 30) / 3
    assert avg == pytest.approx(expected)    # approx for float comparison!

def test_get_nonexistent_server(monitor):
    with pytest.raises(KeyError, match="not found"):
        monitor.get_server("unknown-server")

def test_empty_average(monitor):
    assert monitor.average_cpu() == 0
'''

with open('pytest_demo/test_server_monitor.py', 'w') as f:
    f.write(test_fixtures)

# Run
result = subprocess.run(
    ['python', '-m', 'pytest', 'pytest_demo/test_server_monitor.py', '-v'],
    capture_output=True, text=True
)
print(result.stdout)

# KEY POINTS:
# 1. @pytest.fixture creates reusable setup
# 2. Just add the fixture NAME as a function parameter — pytest injects it
# 3. Each test gets a FRESH fixture (no state leaking between tests)
# 4. pytest.approx() is essential for comparing floats

### Parametrize — Test Multiple Inputs

In [ ]:
# @pytest.mark.parametrize runs the SAME test with DIFFERENT inputs
# Instead of writing 5 test functions, write 1 with 5 sets of data
#
# Without parametrize:
#   def test_add_positive(): assert add(2, 3) == 5
#   def test_add_negative(): assert add(-1, -1) == -2
#   def test_add_zero():     assert add(0, 0) == 0
#
# With parametrize:
#   @pytest.mark.parametrize("a, b, expected", [(2,3,5), (-1,-1,-2), (0,0,0)])
#   def test_add(a, b, expected): assert add(a, b) == expected

test_param = '''import pytest
from calculator import add, subtract, divide

# Test add with multiple inputs
@pytest.mark.parametrize("a, b, expected", [
    (2, 3, 5),         # positive + positive
    (-1, -1, -2),      # negative + negative
    (-1, 1, 0),        # negative + positive
    (0, 0, 0),         # zeros
    (100, 200, 300),   # large numbers
    (1.5, 2.5, 4.0),   # floats
])
def test_add(a, b, expected):
    assert add(a, b) == expected

# Test divide with multiple inputs
@pytest.mark.parametrize("a, b, expected", [
    (10, 2, 5.0),
    (7, 2, 3.5),
    (0, 5, 0.0),
    (1, 3, pytest.approx(0.3333, abs=0.001)),    # approx for repeating decimals
])
def test_divide(a, b, expected):
    assert divide(a, b) == expected

# Test server health classification logic
@pytest.mark.parametrize("cpu, memory, expected_status", [
    (30, 40, "healthy"),       # low usage
    (70, 75, "warning"),       # medium usage
    (95, 90, "critical"),      # high usage
    (50, 50, "healthy"),       # borderline
    (91, 20, "critical"),      # cpu spike
    (20, 91, "critical"),      # memory spike
])
def test_classify_server(cpu, memory, expected_status):
    # Simple classification logic
    if cpu > 90 or memory > 90:
        status = "critical"
    elif cpu > 70 or memory > 70:
        status = "warning"
    else:
        status = "healthy"
    assert status == expected_status
'''

with open('pytest_demo/test_parametrize.py', 'w') as f:
    f.write(test_param)

# Run
result = subprocess.run(
    ['python', '-m', 'pytest', 'pytest_demo/test_parametrize.py', '-v'],
    capture_output=True, text=True
)
print(result.stdout)

# KEY POINT: parametrize is PERFECT for testing edge cases
# Each parameter set creates a SEPARATE test (you can see which one fails)

### Testing ML Code

In [ ]:
# Testing ML code is DIFFERENT from testing regular code
# You can't always test for EXACT values (models are probabilistic)
#
# What to test in ML:
#   1. Data shapes — correct number of rows/columns
#   2. Data types — features are numeric, target is categorical
#   3. No nulls — data is clean after preprocessing
#   4. Model loads — saved model can be loaded
#   5. Predictions work — model returns expected format
#   6. Accuracy threshold — model meets minimum accuracy
#   7. Pipeline consistency — same input → same output

# Create an ML module
ml_module = '''import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import joblib

def create_sample_data(n=500, random_state=42):
    """Generate server health training data."""
    np.random.seed(random_state)
    X = pd.DataFrame({
        "cpu": np.random.randint(10, 100, n),
        "memory": np.random.randint(15, 95, n),
        "disk": np.random.randint(10, 90, n),
        "error_count": np.random.randint(0, 50, n)
    })
    score = (0.3*X["cpu"] + 0.25*X["memory"] + 0.15*X["disk"] +
             0.8*X["error_count"] + np.random.normal(0, 10, n))
    y = (score > 50).astype(int)
    return X, y

def preprocess_data(X, y, test_size=0.2, random_state=42):
    """Split data into train/test sets."""
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

def train_model(X_train, y_train, n_estimators=100, max_depth=10):
    """Train a RandomForest pipeline."""
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", RandomForestClassifier(
            n_estimators=n_estimators, max_depth=max_depth, random_state=42))
    ])
    pipe.fit(X_train, y_train)
    return pipe

def save_model(model, path):
    """Save model to disk."""
    joblib.dump(model, path)

def load_model(path):
    """Load model from disk."""
    return joblib.load(path)
'''

with open('pytest_demo/ml_pipeline.py', 'w') as f:
    f.write(ml_module)

print('ML module created: pytest_demo/ml_pipeline.py')
print('Now let\'s write comprehensive tests for it...')

In [ ]:
# Comprehensive ML tests

ml_tests = '''import pytest
import numpy as np
import pandas as pd
import os
from ml_pipeline import create_sample_data, preprocess_data, train_model, save_model, load_model

# --- FIXTURES ---

@pytest.fixture
def sample_data():
    """Create sample data for testing."""
    return create_sample_data(n=200, random_state=42)

@pytest.fixture
def split_data(sample_data):
    """Split sample data into train/test."""
    X, y = sample_data
    return preprocess_data(X, y)

@pytest.fixture
def trained_model(split_data):
    """Train a model for testing."""
    X_train, X_test, y_train, y_test = split_data
    return train_model(X_train, y_train)

# --- DATA TESTS ---

def test_data_shape(sample_data):
    """Data should have correct number of rows and columns."""
    X, y = sample_data
    assert X.shape == (200, 4), f"Expected (200, 4), got {X.shape}"
    assert len(y) == 200

def test_data_columns(sample_data):
    """Data should have the expected feature columns."""
    X, y = sample_data
    expected_cols = ["cpu", "memory", "disk", "error_count"]
    assert list(X.columns) == expected_cols

def test_data_types(sample_data):
    """Features should be numeric."""
    X, y = sample_data
    for col in X.columns:
        assert X[col].dtype in [np.int32, np.int64, np.float64], f"{col} is not numeric"

def test_no_nulls(sample_data):
    """Data should have no null values."""
    X, y = sample_data
    assert X.isnull().sum().sum() == 0, "Features contain nulls!"
    assert pd.Series(y).isnull().sum() == 0, "Target contains nulls!"

def test_target_classes(sample_data):
    """Target should only contain 0 and 1."""
    X, y = sample_data
    assert set(np.unique(y)).issubset({0, 1})

def test_feature_ranges(sample_data):
    """Features should be in expected ranges."""
    X, y = sample_data
    assert X["cpu"].min() >= 0
    assert X["cpu"].max() <= 100
    assert X["error_count"].min() >= 0

# --- SPLIT TESTS ---

def test_split_sizes(split_data):
    """Train/test split should have correct proportions."""
    X_train, X_test, y_train, y_test = split_data
    total = len(X_train) + len(X_test)
    assert total == 200
    assert len(X_test) == 40    # 20% of 200
    assert len(X_train) == 160  # 80% of 200

def test_split_no_data_leakage(split_data):
    """Train and test should have no overlapping indices."""
    X_train, X_test, y_train, y_test = split_data
    train_idx = set(X_train.index)
    test_idx = set(X_test.index)
    assert len(train_idx & test_idx) == 0, "Data leakage detected!"

# --- MODEL TESTS ---

def test_model_predicts(trained_model, split_data):
    """Model should return predictions."""
    X_train, X_test, y_train, y_test = split_data
    predictions = trained_model.predict(X_test)
    assert len(predictions) == len(X_test)

def test_model_prediction_format(trained_model, split_data):
    """Predictions should be 0 or 1."""
    X_train, X_test, y_train, y_test = split_data
    predictions = trained_model.predict(X_test)
    assert set(np.unique(predictions)).issubset({0, 1})

def test_model_accuracy_threshold(trained_model, split_data):
    """Model should meet minimum accuracy (CI/CD gate)."""
    X_train, X_test, y_train, y_test = split_data
    accuracy = trained_model.score(X_test, y_test)
    assert accuracy >= 0.75, f"Accuracy {accuracy:.2%} below 75% threshold!"

def test_model_probability_output(trained_model, split_data):
    """Model should output valid probabilities."""
    X_train, X_test, y_train, y_test = split_data
    probas = trained_model.predict_proba(X_test)
    assert probas.shape == (len(X_test), 2)    # 2 classes
    assert np.all(probas >= 0) and np.all(probas <= 1)
    assert np.allclose(probas.sum(axis=1), 1.0)    # rows sum to 1

def test_model_deterministic(trained_model, split_data):
    """Same input should give same output (reproducibility)."""
    X_train, X_test, y_train, y_test = split_data
    pred1 = trained_model.predict(X_test)
    pred2 = trained_model.predict(X_test)
    assert np.array_equal(pred1, pred2), "Model is not deterministic!"

# --- SAVE/LOAD TESTS ---

def test_model_save_load(trained_model, split_data, tmp_path):
    """Saved model should produce same predictions when loaded."""
    X_train, X_test, y_train, y_test = split_data

    # Save
    model_path = str(tmp_path / "test_model.pkl")
    save_model(trained_model, model_path)
    assert os.path.exists(model_path)

    # Load
    loaded = load_model(model_path)

    # Compare predictions
    pred_original = trained_model.predict(X_test)
    pred_loaded = loaded.predict(X_test)
    assert np.array_equal(pred_original, pred_loaded), "Loaded model gives different predictions!"
'''

with open('pytest_demo/test_ml_pipeline.py', 'w') as f:
    f.write(ml_tests)

# Run
result = subprocess.run(
    ['python', '-m', 'pytest', 'pytest_demo/test_ml_pipeline.py', '-v'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:] if result.stderr else 'none')

### Mocking — Faking External Dependencies

In [ ]:
# Mocking = replacing real objects with FAKE ones during tests
#
# Why mock?
# - Your code calls an API → you don't want to hit the real API in tests
# - Your code reads from a database → you don't want to need a real DB
# - Your code loads a 2GB model → you don't want tests to be slow
#
# Analogy:
# Testing a fire alarm system:
# Without mocking = start a real fire every time you test (dangerous!)
# With mocking = press the test button (fake smoke, real alarm)

# Module that depends on external service
deploy_module = '''import requests
import json

def get_server_status(server_id):
    """Call monitoring API to get server status."""
    response = requests.get(f"http://monitoring-api:8000/server/{server_id}")
    if response.status_code == 200:
        return response.json()
    raise ConnectionError(f"Failed to get status for {server_id}")

def should_deploy(server_id):
    """Check if it is safe to deploy to this server."""
    status = get_server_status(server_id)
    if status["cpu"] > 80:
        return False, "CPU too high for deployment"
    if status["status"] == "critical":
        return False, "Server is in critical state"
    return True, "Safe to deploy"

def load_config(filepath):
    """Load deployment config from file."""
    with open(filepath, "r") as f:
        return json.load(f)
'''

with open('pytest_demo/deploy_checker.py', 'w') as f:
    f.write(deploy_module)

# Tests with mocking
mock_tests = '''import pytest
from unittest.mock import patch, MagicMock, mock_open
from deploy_checker import get_server_status, should_deploy, load_config

# Mock the HTTP request
class TestShouldDeploy:

    @patch("deploy_checker.get_server_status")
    def test_deploy_when_healthy(self, mock_status):
        """Should allow deploy when server is healthy."""
        # Configure the mock to return fake data
        mock_status.return_value = {"cpu": 30, "memory": 40, "status": "healthy"}

        can_deploy, reason = should_deploy("web-01")
        assert can_deploy is True
        assert reason == "Safe to deploy"

    @patch("deploy_checker.get_server_status")
    def test_deny_deploy_high_cpu(self, mock_status):
        """Should deny deploy when CPU is too high."""
        mock_status.return_value = {"cpu": 95, "memory": 40, "status": "warning"}

        can_deploy, reason = should_deploy("web-01")
        assert can_deploy is False
        assert "CPU" in reason

    @patch("deploy_checker.get_server_status")
    def test_deny_deploy_critical(self, mock_status):
        """Should deny deploy when server is critical."""
        mock_status.return_value = {"cpu": 50, "memory": 60, "status": "critical"}

        can_deploy, reason = should_deploy("web-01")
        assert can_deploy is False
        assert "critical" in reason

# Mock file operations
class TestLoadConfig:

    def test_load_config(self):
        """Should load JSON config from file."""
        fake_config = \'{"app": "web", "replicas": 3}\'
        with patch("builtins.open", mock_open(read_data=fake_config)):
            config = load_config("fake/path/config.json")
            assert config["app"] == "web"
            assert config["replicas"] == 3
'''

with open('pytest_demo/test_deploy_checker.py', 'w') as f:
    f.write(mock_tests)

# Run
result = subprocess.run(
    ['python', '-m', 'pytest', 'pytest_demo/test_deploy_checker.py', '-v'],
    capture_output=True, text=True
)
print(result.stdout)

# KEY POINTS:
# 1. @patch replaces a function with a mock during the test
# 2. mock.return_value controls what the mock returns
# 3. Tests run WITHOUT actually calling the API or reading files
# 4. Each test is fast, reliable, and independent

### Running Tests — Command Reference

In [ ]:
# Pytest command cheat sheet:
#
# pytest                          # run ALL tests in current directory
# pytest -v                       # verbose (show each test name)
# pytest -v test_calculator.py    # run tests in one file
# pytest -v -k "test_add"         # run tests matching a pattern
# pytest -v -k "not test_divide"  # skip tests matching a pattern
# pytest -x                       # stop at first failure
# pytest --tb=short               # shorter traceback on failure
# pytest --co                     # collect (list) tests without running
# pytest -v --durations=5         # show 5 slowest tests
#
# Coverage (install: pip install pytest-cov):
# pytest --cov=my_module          # show code coverage percentage
# pytest --cov=my_module --cov-report=html   # HTML coverage report
#
# In CI/CD (Azure DevOps, GitHub Actions):
# pytest --junitxml=results.xml   # generate XML report for CI

print('Pytest Command Reference:')
print()
commands = [
    ('pytest', 'Run all tests'),
    ('pytest -v', 'Verbose output'),
    ('pytest -x', 'Stop at first failure'),
    ('pytest -k "add"', 'Run tests matching pattern'),
    ('pytest --tb=short', 'Shorter error output'),
    ('pytest --cov=module', 'Show code coverage'),
    ('pytest --junitxml=report.xml', 'CI/CD report'),
]
for cmd, desc in commands:
    print(f'  {cmd:40s} → {desc}')

In [ ]:
# Let's run ALL tests at once to see the full picture

result = subprocess.run(
    ['python', '-m', 'pytest', 'pytest_demo/', '-v', '--tb=short'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('Some tests may have failed — check output above')

In [ ]:
# Practice Questions:

# 1. Write tests for a function parse_log_line(line) that:
#    - Extracts timestamp, level, and message from a log line
#    - Format: "2024-01-15 10:30:00 ERROR Database connection failed"
#    - Test: valid line, missing fields, empty string

# 2. Write parametrized tests for a function classify_cpu(value):
#    - 0-50: "normal", 51-80: "high", 81-100: "critical"
#    - Test at least 8 different values including boundaries

# 3. Create a fixture that:
#    - Creates a temporary CSV file with server data
#    - Yields the file path
#    - Cleans up the file after the test
#    Write 3 tests that use this fixture

# 4. Write tests for the model save/load cycle:
#    - Train a model, save it, load it
#    - Verify loaded model has same accuracy (within tolerance)
#    - Verify predictions match between original and loaded

### DevOps / AI-MLOps Practice

In [ ]:
# Testing is the FOUNDATION of CI/CD — no tests, no automatic deployment!

# Exercise 1: Config Validator Tests
# Write a function validate_k8s_config(config_dict) that checks:
#   - 'replicas' exists and is > 0
#   - 'image' exists and contains ':' (tag)
#   - 'port' exists and is between 1-65535
#   Raises ValueError with clear message for each violation
# Write tests using pytest.raises and parametrize for:
#   - Valid config → passes
#   - Missing replicas → ValueError
#   - replicas=0 → ValueError
#   - Port out of range → ValueError
#   - Missing image tag → ValueError

# Exercise 2: ML Pipeline CI/CD Tests
# Write a test suite that acts as a CI/CD GATE:
#   test_data_quality()    → no nulls, correct columns, valid ranges
#   test_model_accuracy()  → accuracy >= 80% (fail the build if not!)
#   test_model_latency()   → prediction takes < 100ms
#   test_model_size()      → saved model file < 50MB
# If ANY test fails, the CI/CD pipeline should STOP deployment

# Exercise 3: API Integration Tests
# Write tests for the FastAPI prediction API (from Day 34):
#   test_health_endpoint()    → GET /healthz returns 200
#   test_predict_valid()      → POST /predict with valid data returns prediction
#   test_predict_invalid()    → POST /predict with bad data returns 422
#   test_predict_batch()      → POST /predict/batch works correctly
# Use unittest.mock to mock the model (don't need a real model for API tests)

# Why this matters for me as a DevOps/MLOps engineer:
# 1. CI/CD pipeline: push code → pytest runs → if all pass → deploy
# 2. ML-specific tests prevent deploying bad models to production
# 3. Config validators catch errors BEFORE they hit Kubernetes
# 4. Mocking lets you test deployment logic without a real cluster
# 5. pytest --junitxml generates reports for Azure DevOps/GitHub Actions

# Clean up
import shutil
if os.path.exists('pytest_demo'):
    shutil.rmtree('pytest_demo')
    print('Demo files cleaned up!')